# Gradient Descent Model:-

- Here is the complete BigQuery ML (BQML) pipeline to train a Gradient Boosted Decision Trees (XGBoost) model on the StudentsPerformance.csv dataset to predict student test outcomes (e.g., predicting math_score based on reading/writing scores, demographics, and prep courses).

# Key Concepts for Gradient Boosting in BQML:
1. **Model Type**: Under the hood, BQML implements Gradient Boosting using XGBoost with BOOSTED_TREE_REGRESSOR (for numeric score prediction) or BOOSTED_TREE_CLASSIFIER (for pass/fail classification).
2. **Sequential Trees (Boosting)**: Unlike Random Forest which builds trees independently in parallel, Gradient Boosting builds trees sequentially—each new tree corrects the residual errors made by the previous trees.
3. **Key Hyperparameters**:
    - **max_iterations**: Number of boosting rounds (trees built sequentially, e.g., 20–50).
    - **learn_rate**: Step size shrinkage used to prevent overfitting (e.g., 0.1 or 0.05).
    - **max_tree_depth**: Maximum depth of each decision tree (e.g., 4–6).

# Step by step Complete Gradient Descent Model Guide:-

- **Step 1**: Create Schema / Dataset
```sql
CREATE SCHEMA IF NOT EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox`
OPTIONS(
  location = 'US'
);
```
> gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_students_performance

---

- **Step 2**: Cleanse and Rename Columns -> Raw student performance CSVs often have spaces and slashes in column names (e.g., race/ethnicity, math score). We standardize and clean them:

```sql
CREATE OR REPLACE TABLE `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_students_performance` AS
SELECT
  -- Categorical Features
  CAST(gender AS STRING) AS gender,
  CAST(race_ethnicity AS STRING) AS race_ethnicity,
  CAST(`parental level of education` AS STRING) AS parental_education,
  CAST(lunch AS STRING) AS lunch_type,
  CAST(`test preparation course` AS STRING) AS test_prep_course,
  
  -- Numeric Exam Scores
  CAST(`reading score` AS FLOAT64) AS reading_score,
  CAST(`writing score` AS FLOAT64) AS writing_score,
  
  -- Target Label (Math Score)
  CAST(`math score` AS FLOAT64) AS math_score
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_students_performance`;
```

---

- **Step 3**: Train the Gradient Boosted Tree Model (XGBoost)

```sql
CREATE OR REPLACE MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.student_score_gb_model`
OPTIONS (
  model_type = 'BOOSTED_TREE_REGRESSOR',
  input_label_cols = ['math_score'],
  
  -- Gradient Boosting Hyperparameters
  max_iterations = 30,
  learn_rate = 0.1,
  max_tree_depth = 4,
  subsample = 0.85,
  
  -- 80% train / 20% test split
  data_split_method = 'RANDOM',
  data_split_eval_fraction = 0.2
) AS
SELECT
  gender,
  race_ethnicity,
  parental_education,
  lunch_type,
  test_prep_course,
  reading_score,
  writing_score,
  math_score
FROM
  `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_students_performance`;
```

---

- **Step 4**: Evaluate Model Performance (R², MAE, RMSE) -> Evaluate how close the gradient booster's score predictions are to actual test scores:

```sql
SELECT
  *
FROM
  ML.EVALUATE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.student_score_gb_model`);
```

---


- **Step 5**: Feature Importance (What influences Math scores most?) -> Identify which factors (e.g., reading_score, writing_score, test_prep_course) have the highest predictive gain in the boosted trees.
```sql
SELECT
  feature,
  importance_gain,
  importance_weight
FROM
  ML.FEATURE_IMPORTANCE(MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.student_score_gb_model`)
ORDER BY
  importance_gain DESC;
```

---


- **Step 6**: Predict Math Scores for New Students -> Predict expected math scores for two hypothetical student profiles:
    - **Student 1**: Completed test prep course, high reading/writing scores (90+), standard lunch.
    - **Student 2**: No test prep, average reading/writing scores (60), free/reduced lunch.

```sql
SELECT
  ROUND(predicted_math_score, 1) AS predicted_math_score,
  gender,
  lunch_type,
  test_prep_course,
  reading_score,
  writing_score
FROM
  ML.PREDICT(
    MODEL `gcp-aiml-sandbox-rahuld-1234.student_sandbox.student_score_gb_model`,
    (
      -- Profile 1: High reader/writer with prep course
      SELECT 
        CAST('female' AS STRING) AS gender,
        CAST('group D' AS STRING) AS race_ethnicity,
        CAST("bachelor's degree" AS STRING) AS parental_education,
        CAST('standard' AS STRING) AS lunch_type,
        CAST('completed' AS STRING) AS test_prep_course,
        CAST(92.0 AS FLOAT64) AS reading_score,
        CAST(95.0 AS FLOAT64) AS writing_score
      UNION ALL
      -- Profile 2: Average scores, no prep course
      SELECT 
        CAST('male' AS STRING) AS gender,
        CAST('group B' AS STRING) AS race_ethnicity,
        CAST('some high school' AS STRING) AS parental_education,
        CAST('free/reduced' AS STRING) AS lunch_type,
        CAST('none' AS STRING) AS test_prep_course,
        CAST(62.0 AS FLOAT64) AS reading_score,
        CAST(58.0 AS FLOAT64) AS writing_score
    )
  );
```

---


- **Step 7**: Clean Up Resources
```sql
DROP MODEL IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.student_score_gb_model`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.cleaned_students_performance`;
DROP TABLE IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox.raw_students_performance`;
DROP SCHEMA IF EXISTS `gcp-aiml-sandbox-rahuld-1234.student_sandbox` CASCADE;
```

---